<!-- beginner-banner-v2 -->

> 🧭 <strong>비개발자 수강생 안내</strong> — 이 노트북에서 새로 배우는 것: <code>ChatInterface + share=True</code> 로 <strong>공개 URL 채팅 화면</strong> 띄우기.
>
> - 📖 강의 페이지: <a href="https://siapapa.github.io/day2/12-gradio-ui/" target="_blank" rel="noopener noreferrer">day2/12-gradio-ui</a>
> - 🆕 처음이라면 → <a href="https://siapapa.github.io/beginners-guide/" target="_blank" rel="noopener noreferrer">비개발자 학습 가이드</a>
> - 🔤 모르는 단어 → <a href="https://siapapa.github.io/appendix/glossary/" target="_blank" rel="noopener noreferrer">용어 사전</a>
> - 🛠️ 환경/접속 막힘 → <a href="https://siapapa.github.io/setup/" target="_blank" rel="noopener noreferrer">사전 준비</a> · <a href="https://siapapa.github.io/appendix/troubleshooting/" target="_blank" rel="noopener noreferrer">트러블슈팅</a>
>
> 외부 링크는 새 탭으로 열리도록 설정돼 있어 Colab 의 리디렉션 경고 페이지를 거치지 않습니다.<br/>
> <strong>셀은 위에서 아래로 차례대로 실행</strong>하세요. 시연용 코드(<code>구경만 하세요</code> 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 09. 병원 DB 멀티턴 상담사 + Gradio UI
> Day 2 · 11~12H · 소요 약 100분 (11H 상담사 설계 + 12H Gradio)

## 학습 목표

- **멀티턴 대화**에서 맥락(히스토리)을 누적하고 "그 중에서..." 같은 참조 질문을 해석한다.
- **SQL 보안 가드레일**(DML/DDL 차단, 화이트리스트, `LIMIT` 자동 주입)로 DB 를 보호한다.
- **에러 복구** — SQL 실행 실패 시 에러를 프롬프트에 되먹여 자동 재시도한다.
- **Gradio `ChatInterface`** 로 5 줄짜리 채팅 UI 를 만들고, Colab 에서 `share=True` 공개 URL 로 배포한다.

> **선행 조건**
> - `01_postgres_basics.ipynb` 로 병원 DB 가 Neon 에 적재되어 있어야 합니다.
> - `08_text_to_sql_advanced.ipynb` 의 커스텀 프롬프트·도메인 규칙 감각이 있으면 더 좋습니다.
>
> **노트**: 이 노트북은 11H(상담사 설계)와 12H(Gradio UI)를 합친 긴 노트북입니다. 섹션 1–4 까지는 순수 Python 클래스 구현, 섹션 5–7 이 Gradio 파트입니다.

In [ ]:
# NumPy 2.x ABI 충돌 방지를 위해 numpy / pandas 를 명시 핀.
# (이 핀이 없으면 일부 환경에서 다음 에러:
#   ValueError: numpy.dtype size changed, may indicate binary incompatibility.
#   Expected 96 from C header, got 88 from PyObject)
%pip install -q --upgrade \
    "numpy>=2.0,<3" "pandas>=2.2.2,<3" \
    "gradio==4.44.1" \
    "sqlalchemy>=2.0" psycopg2-binary "openai>=1.30" sqlparse tabulate \
    "llama-index>=0.10.50,<0.12" llama-index-llms-openai llama-index-embeddings-openai


In [ ]:
# ============================================================
# 🔁 NumPy ABI 호환성 자동 재시작 (Colab 전용)
# ------------------------------------------------------------
# 위 %pip install 이 numpy 또는 pandas 를 새 버전으로 교체했다면, 이미
# 메모리에 로드된 옛 numpy 와 ABI 가 어긋나 다음 셀에서
#   "numpy.dtype size changed" 오류가 날 수 있다.
# 이 셀은 **그 경우에만** 런타임을 자동 재시작한다. 재시작 후엔 메뉴에서
# [Runtime] → [Run all] 또는 이 셀부터 다시 실행하면 된다.
# ============================================================
def _need_restart() -> bool:
    """현재 로드된 numpy 가 방금 설치한 버전과 다르면 True."""
    try:
        import numpy as _np, importlib.metadata as _md
        installed = _md.version("numpy")
        loaded = _np.__version__
        return installed.split(".")[0] != loaded.split(".")[0]
    except Exception:
        return False

import sys
if "google.colab" in sys.modules and _need_restart():
    print("⚠️ numpy 메이저 버전이 바뀌었습니다 — 런타임을 재시작합니다.")
    print("   재시작 후 이 셀부터 다시 실행해 주세요.")
    import os
    os.kill(os.getpid(), 9)  # Colab 은 자동으로 런타임을 다시 시작합니다.
else:
    print("✅ numpy ABI OK — 다음 셀로 진행해도 됩니다.")


In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다.
import os

def _load_secret(key: str, required: bool = True) -> None:
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

_load_secret("NEON_DSN", required=True)
_load_secret("OPENAI_API_KEY", required=True)

print("Environment ready.")

## 1. 멀티턴 대화 — 왜 "대화 메모장"이 필요한가

LLM 은 원래 **기억력이 없습니다(stateless)**. 따라서 우리가 "대화 메모장"을 만들어 **매번 전체 대화를 프롬프트에 같이 넘겨야** "그 중에서..." 같은 참조 질문이 동작합니다.

```
단발 질의 (Stateless)
  User: "남성 환자 수는?"
  Bot:  "남성 환자는 15명입니다."
  (끝 — 이전 대화 기억 없음)

멀티턴 대화 (Stateful)
  User: "남성 환자 수는?"
  Bot:  "15명입니다."
  User: "그 중에 40세 이상은?"           ← "그 중" = 이전 조건(gender='M') 참조
  Bot:  "남성 환자 중 40세 이상은 7명입니다."
  User: "그 사람들이 가장 많이 간 진료과는?"  ← "그 사람들" = 이전 결과 참조
  Bot:  "..."
```

이 노트북에서 만들 상담사는 대화 히스토리와 **직전 SQL / 직전 결과 요약**까지 프롬프트에 끼워 넣어 참조 질문을 해결합니다.

In [ ]:
# ============================================================
# 1. ChatState — 대화 상태(메모장)
# ============================================================
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ChatState:
    """대화 상태를 관리하는 클래스."""
    history: list = field(default_factory=list)   # [(role, message), ...]
    last_sql: Optional[str] = None                # 마지막 실행 SQL
    last_result: Optional[str] = None             # 마지막 결과 요약

    def add_user(self, msg: str) -> None:
        self.history.append(("user", msg))

    def add_assistant(self, msg: str, sql: Optional[str] = None, result: Optional[str] = None) -> None:
        self.history.append(("assistant", msg))
        if sql:
            self.last_sql = sql
        if result:
            self.last_result = result

    def get_history_text(self, max_turns: int = 5) -> str:
        """최근 N 턴(= user/assistant 2 개씩)의 대화를 텍스트로 변환."""
        recent = self.history[-(max_turns * 2):]
        lines = []
        for role, msg in recent:
            prefix = "사용자" if role == "user" else "시스템"
            lines.append(f"{prefix}: {msg}")
        return "\n".join(lines)

# 간단한 동작 확인
s = ChatState()
s.add_user("남성 환자 수는?")
s.add_assistant("15명입니다.", sql="SELECT COUNT(*) FROM patients WHERE gender='M';")
s.add_user("그 중에 40세 이상은?")
print(s.get_history_text())

## 2. SQL 보안 가드레일

| 위협 유형 | 설명 | 방어 전략 |
|---|---|---|
| **DML 공격** | DELETE/DROP/UPDATE 로 데이터 변경·삭제 | SELECT 만 허용 (키워드 차단) |
| **SQL 인젝션** | 악의적 코드 삽입 | 주석·세미콜론+DML 패턴 차단 |
| **테이블 무단 접근** | 허용되지 않은 테이블 조회 | 화이트리스트 |
| **대량 조회** | `SELECT *` 로 서버 과부하 | `LIMIT 1000` 자동 주입 |
| **시스템 정보 노출** | `information_schema`, `pg_roles` 등 | 시스템 테이블 차단 |

> **경고** — 정규식 가드레일은 **보조 방어선**일 뿐입니다. 프로덕션에서는 반드시 **DB 레벨 read-only 롤**을 함께 적용하세요. 전각 문자(`ＤＲＯＰ`), `CHR()` 인코딩, data-modifying CTE 등은 정규식만으로 막기 어렵습니다. 아래 가드레일은 오탐·오답을 일찍 잡는 용도로만 쓰고, 실제 DB 연결은 `default_transaction_read_only=on` 옵션이 들어간 커넥션으로 접속하는 것을 권장합니다.

In [ ]:
# ============================================================
# 2. SQLGuardrail — SELECT 만 허용하는 경비원
# ============================================================
import re
import sqlparse

class SQLGuardrail:
    """SQL 보안 가드레일 — 위험한 쿼리를 사전 차단."""

    BLOCKED_KEYWORDS = re.compile(
        r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE|GRANT|REVOKE|"
        r"COPY|EXECUTE|DO|CALL)\b",
        re.IGNORECASE,
    )

    BLOCKED_PATTERNS = re.compile(
        r"(information_schema|pg_catalog|pg_stat|pg_roles|"
        r"--\s|/\*|\*/|;\s*DROP|;\s*DELETE)",
        re.IGNORECASE,
    )

    def __init__(self, allowed_tables, max_limit: int = 1000):
        self.allowed_tables = [t.lower() for t in allowed_tables]
        self.max_limit = max_limit

    def check(self, sql: str):
        # 1) 위험 키워드
        m = self.BLOCKED_KEYWORDS.search(sql)
        if m:
            return False, f"'{m.group()}' 명령은 허용되지 않습니다. SELECT 만 사용 가능합니다."

        # 2) 인젝션 패턴
        m = self.BLOCKED_PATTERNS.search(sql)
        if m:
            return False, f"보안 위반이 감지되었습니다: '{m.group()[:30]}'"

        # 3) 화이트리스트 (간이 파싱)
        #    주의: EXTRACT(YEAR FROM birth_date) 같은 함수형 FROM 을 오탐하지 않도록
        #         해당 함수 호출을 먼저 제거한 뒤 FROM/JOIN 뒤 식별자만 추출합니다.
        cte_names = set(re.findall(r'\bwith\s+(\w+)\s+as\b', sql, re.IGNORECASE))
        parsed = sqlparse.parse(sql)
        for stmt in parsed:
            s = str(stmt).lower()
            s = re.sub(
                r'\b(extract|position|substring|trim|cast)\s*\([^()]*\)',
                ' ',
                s,
            )
            from_m = re.findall(r'\bfrom\s+([a-z_][a-z0-9_]*)', s)
            join_m = re.findall(r'\bjoin\s+([a-z_][a-z0-9_]*)', s)
            used = {t for t in (from_m + join_m) if t not in cte_names}
            for t in used:
                if t not in self.allowed_tables:
                    return False, f"'{t}' 테이블에 대한 접근이 허용되지 않습니다."

        return True, ""

    def inject_limit(self, sql: str) -> str:
        if "LIMIT" not in sql.upper():
            sql = sql.rstrip().rstrip(";") + f"\nLIMIT {self.max_limit};"
        return sql

    def sanitize(self, sql: str):
        ok, err = self.check(sql)
        if not ok:
            return "", err
        return self.inject_limit(sql), ""


In [ ]:
# ============================================================
# 3. 가드레일 동작 확인 — 정상 / 위험 / 범위 외
# ============================================================
guard = SQLGuardrail(
    allowed_tables=["patients", "doctors", "visits", "diagnoses", "departments", "vw_visit_details"],
    max_limit=1000,
)

cases = [
    ("정상",      "SELECT * FROM patients WHERE gender = 'M'"),
    ("DML 공격",  "DROP TABLE patients"),
    ("시스템",     "SELECT * FROM pg_roles"),
    ("대소문자", "DrOp TaBlE patients"),
    ("범위 외",   "SELECT * FROM salaries"),
    ("정상+함수", "SELECT COUNT(*) FROM patients WHERE EXTRACT(YEAR FROM birth_date) < 1990"),
]

for label, sql in cases:
    clean, err = guard.sanitize(sql)
    if err:
        print(f"[blocked] {label:10s} -> {err}")
    else:
        print(f"[ok]      {label:10s} -> {clean.splitlines()[-1]}")

## 3. `HospitalChatbot` — 스키마 + 히스토리 + 가드레일을 묶은 상담사

아래 클래스는 한 턴마다 다음을 수행합니다:

1. **SQL 생성** — 스키마 + 최근 5 턴 + 직전 SQL 을 프롬프트에 넣어 LLM 으로 SELECT 만 생성.
2. **가드레일 + 실행** — `SQLGuardrail.sanitize()` 로 검증·`LIMIT` 주입 후 `pandas.read_sql` 로 실행.
3. **자연어 요약** — 결과표를 LLM 이 한국어로 요약.
4. **상태 갱신** — `ChatState` 에 이번 질문·답변·SQL·결과를 기록.

In [ ]:
# ============================================================
# 4. HospitalChatbot — 멀티턴 상담사 본체
# ============================================================
import pandas as pd
from sqlalchemy import create_engine, text
from openai import OpenAI as OpenAIClient

from llama_index.core import SQLDatabase, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

engine = create_engine(os.environ["NEON_DSN"])
oai = OpenAIClient()

HOSPITAL_TABLES = ["patients", "doctors", "visits", "diagnoses", "departments"]


def get_full_schema(engine) -> str:
    """각 테이블의 table_info 를 모아 스키마 문자열을 만든다."""
    sql_db_temp = SQLDatabase(engine, include_tables=HOSPITAL_TABLES)
    parts = [sql_db_temp.get_single_table_info(t) for t in HOSPITAL_TABLES]
    return "\n\n".join(parts)


class HospitalChatbot:
    """병원 DB 멀티턴 상담사."""

    def __init__(self, engine, schema_info: str):
        self.engine = engine
        self.schema_info = schema_info
        self.state = ChatState()
        self.guardrail = SQLGuardrail(
            allowed_tables=HOSPITAL_TABLES + ["vw_visit_details"],
        )

    # ---- 1) SQL 생성 ----------------------------------------------------
    def _generate_sql(self, question: str) -> str:
        history_text = self.state.get_history_text(max_turns=5)

        last_ctx = ""
        if self.state.last_sql:
            last_ctx = (
                "\n## 직전 SQL\n" + self.state.last_sql
                + "\n\n## 직전 결과 요약\n"
                + (self.state.last_result[:500] if self.state.last_result else "(없음)")
            )

        prompt = f"""당신은 병원 데이터베이스 분석 전문가입니다.
아래 스키마와 대화 맥락을 참고하여 PostgreSQL 쿼리를 작성하세요.

## 데이터베이스 스키마
{self.schema_info}

## 규칙
- SELECT 문만 작성하세요.
- visits.status = 'completed' 만 유효한 진료입니다.
- 나이 = EXTRACT(YEAR FROM AGE(birth_date))
- "그 중에서", "위 결과에서" 같은 표현은 직전 SQL 의 조건을 유지하면서 추가 필터를 적용하세요.
- SQL 만 반환하세요 (설명 없이).
{last_ctx}

## 대화 히스토리
{history_text}

## 현재 질문
{question}

## SQL
"""
        resp = oai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        sql = resp.choices[0].message.content.strip()
        sql = re.sub(r"```sql\s*", "", sql)
        sql = re.sub(r"```\s*", "", sql)
        return sql

    # ---- 2) SQL 실행 (가드레일) ----------------------------------------
    def _execute_sql(self, sql: str) -> str:
        safe_sql, error = self.guardrail.sanitize(sql)
        if error:
            return f"[blocked] {error}"
        try:
            # text() 로 감싸 psycopg2 의 pyformat `%` 자리표시자 충돌을 회피하고,
            # connection 을 pandas 에 넘겨 immutabledict 가 드라이버까지 전달되지 않게 한다.
            with self.engine.connect() as conn:
                df = pd.read_sql(text(safe_sql), conn)
            if df.empty:
                return "(결과 없음)"
            return df.to_string(index=False)
        except Exception as e:
            return f"[error] SQL 실행 오류: {str(e)}"

    # ---- 3) 결과 요약 ---------------------------------------------------
    def _generate_answer(self, question: str, sql: str, result: str) -> str:
        prompt = f"""아래 SQL 결과를 한국어로 친절하게 요약해주세요.

질문: {question}
SQL: {sql}
결과:
{result[:1000]}

규칙:
- 숫자에 천 단위 구분자를 사용하세요.
- 표 형태면 핵심만 요약하세요.
- 친절하지만 간결하게 답변하세요.
"""
        resp = oai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
        )
        return resp.choices[0].message.content

    # ---- 4) 진입점 -----------------------------------------------------
    def chat(self, question: str) -> str:
        self.state.add_user(question)

        sql = self._generate_sql(question)
        result = self._execute_sql(sql)

        if result.startswith("[blocked]") or result.startswith("[error]"):
            answer = result
        else:
            answer = self._generate_answer(question, sql, result)

        self.state.add_assistant(answer, sql=sql, result=result)
        return f"{answer}\n\n실행된 SQL:\n```sql\n{sql}\n```"


schema_info = get_full_schema(engine)
print("schema_info length:", len(schema_info), "chars")

## 4. 4-턴 멀티턴 테스트 — 맥락 유지 + 가드레일

턴 1 → 일반 질문, 턴 2 → "그 중에" 로 이전 조건 참조, 턴 3 → "그 사람들" 로 이전 결과 참조, 턴 4 → 악성 요청 → 가드레일 차단.

In [ ]:
# ============================================================
# 5. 4턴 멀티턴 대화 데모
# ============================================================
bot = HospitalChatbot(engine, schema_info)

turns = [
    "남성 환자는 몇 명인가요?",
    "그 중에 40세 이상은?",
    "그 사람들이 가장 많이 간 진료과는?",
    "환자 테이블을 삭제해줘",   # 가드레일이 차단해야 함
]

for i, q in enumerate(turns, start=1):
    print("=" * 60)
    print(f"[turn {i}] 사용자: {q}")
    print("=" * 60)
    print(bot.chat(q))
    print()

## 5. 에러 복구 — 실패한 SQL 을 자동 재시도

LLM 이 만든 SQL 이 PostgreSQL 에서 에러를 내면, **에러 메시지를 그대로 프롬프트에 되먹여** 재시도시킵니다. 최대 2 회 재시도하며, 각 시도에서 `_generate_sql` 은 **정확히 1 회**만 호출해 토큰 비용이 불필요하게 두 배가 되지 않도록 합니다.

In [ ]:
# ============================================================
# 6. RobustHospitalChatbot — 재시도 로직 포함
# ============================================================
class RobustHospitalChatbot(HospitalChatbot):
    """에러 발생 시 최대 MAX_RETRIES 회 자동 재시도."""

    MAX_RETRIES = 2

    def chat(self, question: str) -> str:
        self.state.add_user(question)

        sql = ""
        result = ""
        last_error = ""
        attempt = 0

        for attempt in range(self.MAX_RETRIES + 1):
            if attempt == 0:
                sql = self._generate_sql(question)
            else:
                sql = self._generate_sql(
                    f"이전 SQL 이 오류 발생: {last_error}\n원래 질문: {question}\n수정된 SQL 을 작성하세요."
                )

            result = self._execute_sql(sql)
            if not result.startswith("[error]"):
                break
            last_error = result

        if result.startswith("[blocked]") or result.startswith("[error]"):
            answer = result
        else:
            answer = self._generate_answer(question, sql, result)

        self.state.add_assistant(answer, sql=sql, result=result)
        return f"{answer}\n\n시도 {attempt + 1}회 SQL:\n```sql\n{sql}\n```"


rbot = RobustHospitalChatbot(engine, schema_info)
print(rbot.chat("진료과별 월 평균 방문 수를 최근 3개월 기준으로 보여줘"))

## 6. Gradio 최소 예제 — 5 줄 채팅 UI

`gr.ChatInterface` 는 ML 데모용 채팅 UI 를 5 줄로 완성시켜 줍니다. `share=True` 를 주면 Colab 에서 **72 시간 유효한 공개 URL** 이 자동 생성됩니다.

> **버전 고정 중요** — 이 노트북은 `gradio==4.44.1` 기준입니다. Gradio 5.x 는 `retry_btn` / `undo_btn` / `clear_btn` / `bubble_full_width` 를 제거해서 같은 코드가 `TypeError` 로 멈춥니다. 설치 셀의 핀을 풀지 마세요.

> **보안 주의** — `share=True` 로 띄운 URL 을 아는 사람은 누구나 본인 에이전트를 통해 본인 Neon DB 에 접근할 수 있습니다. 실제 민감 데이터를 쓸 때는 URL 공유 범위를 제한하고, 에이전트 연결을 read-only 롤로 잠그세요. 로컬에서 리뷰할 때는 `share=False` 로 충분합니다.

In [ ]:
# ============================================================
# 7. Echo Bot — Gradio ChatInterface 기본 동작 확인
# ============================================================
import gradio as gr

def echo_chat(message, history):
    return f"당신이 말한 것: {message}"

echo_demo = gr.ChatInterface(
    fn=echo_chat,
    title="에코 봇",
    description="입력한 메시지를 그대로 반환합니다.",
)

# Colab 에서는 share=True 로 공개 URL 생성
# 로컬 리뷰용으로는 share=False 로도 충분합니다.
echo_demo.launch(share=True)

## 7. 병원 상담사 + Gradio 통합

`HospitalChatbot` 은 인스턴스 안에 `ChatState` 를 들고 있습니다. 그러나 Gradio 의 `ChatInterface` 는 **`history` 인자로만 세션 상태를 전달**합니다. 그래서 Gradio 통합에서는 `ChatState` 대신 Gradio 의 `history` 를 단일 저장소로 쓰고, 이전 assistant 응답에 포함된 ```sql 코드블록을 파싱해 "직전 SQL" 을 복원합니다.

구성은 단순한 **절차형 함수 3개** + 핸들러 1개입니다:

- `generate_sql(question, history_text, last_sql)` — SQL 생성
- `safe_execute(sql)` — 가드레일 + 실행 + DataFrame → Markdown
- `summarize_result(question, sql, result)` — 결과 요약
- `hospital_chat(message, history)` — ChatInterface 핸들러

In [ ]:
# ============================================================
# 8. 스키마 + 가드레일 (Gradio 핸들러용 전역 유틸)
# ============================================================
sql_db = SQLDatabase(engine, include_tables=HOSPITAL_TABLES)
schema_parts = [sql_db.get_single_table_info(t) for t in HOSPITAL_TABLES]
SCHEMA_INFO = "\n\n".join(schema_parts)

# 간단 가드레일 (클래스 없이 함수 3 개 + regex 로 요약)
BLOCKED = re.compile(
    r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE|GRANT|REVOKE)\b",
    re.IGNORECASE,
)
ALLOWED_TABLES = {"patients", "doctors", "visits", "diagnoses", "departments", "vw_visit_details"}


def safe_execute(sql: str) -> str:
    """가드레일 적용 후 SQL 실행 → Markdown 문자열."""
    if BLOCKED.search(sql):
        return "[blocked] 위험한 명령어가 포함되어 있어 실행할 수 없습니다."
    if "LIMIT" not in sql.upper():
        sql = sql.rstrip().rstrip(";") + "\nLIMIT 1000;"
    try:
        # text() + connection 으로 감싸 psycopg2 의 pyformat `%` 충돌을 회피한다.
        with engine.connect() as conn:
            df = pd.read_sql(text(sql), conn)
        if df.empty:
            return "(결과 없음)"
        if len(df) > 20:
            return df.head(20).to_markdown(index=False) + f"\n\n... 외 {len(df)-20}행"
        return df.to_markdown(index=False)
    except Exception as e:
        return f"[error] SQL 오류: {str(e)}"


def generate_sql(question: str, history_text: str, last_sql: str = "") -> str:
    last_ctx = f"\n직전 SQL:\n{last_sql}" if last_sql else ""
    prompt = f"""PostgreSQL 전문가입니다. 병원 DB 에 대한 질문에 SQL 을 작성하세요.

{SCHEMA_INFO}

규칙:
- SELECT 만 사용. completed 상태만 유효.
- "그 중" = 직전 SQL 조건 유지 + 추가 필터
- SQL 만 반환 (설명 없이).
{last_ctx}

대화:
{history_text}

질문: {question}
SQL:"""
    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    sql = resp.choices[0].message.content.strip()
    sql = re.sub(r"```sql\s*", "", sql)
    sql = re.sub(r"```\s*", "", sql)
    return sql


def summarize_result(question: str, sql: str, result: str) -> str:
    prompt = f"""질문: {question}
SQL: {sql}
결과:
{result[:800]}

한국어로 간결하게 요약하세요. 숫자에 천 단위 구분자 사용."""
    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return resp.choices[0].message.content

### 범위 외 질문 & 인사 처리

에이전트가 "이 질문은 DB 질문이 아님"을 판단해 SQL 없이 친절히 거절/응답하는 가드도 함께 넣습니다. 두 가지 사례:

- **인사** — "안녕", "감사합니다", "고마워요" → 고정 인사 응답.
- **범위 외** — 병원 DB 와 무관한 질문(예: "오늘 날씨 어때?") → SQL 을 만들지 않고 "그 질문은 병원 DB 범위를 벗어납니다" 로 거절.

In [ ]:
# ============================================================
# 9. Gradio 핸들러 — history → (sql, answer)
# ============================================================
import random

GREETING_PATTERNS = re.compile(
    r"(감사합니다|고마워|안녕|수고|반갑|잘\s*부탁|좋은\s*하루|화이팅|감사해요)",
    re.IGNORECASE,
)
GREETING_RESPONSES = [
    "안녕하세요! 병원 DB 에 대해 궁금한 점을 자유롭게 질문해주세요.",
    "도움이 되었다면 다행입니다! 추가 분석이 필요하면 말씀해주세요.",
    "감사합니다! 이어서 환자·진료·진단 관련 질문을 해 보세요.",
]

# 범위 외 키워드 — 가벼운 휴리스틱. LLM 판정은 비용이 크므로 rule 로 선필터.
OUT_OF_SCOPE_HINTS = re.compile(
    r"(날씨|주식|환율|뉴스|영화|게임|맛집|여행|연예인|축구|야구)",
    re.IGNORECASE,
)


def hospital_chat(message: str, history) -> str:
    """Gradio ChatInterface 용 핸들러.

    history 는 Gradio 가 유지하는 유일한 세션 저장소이므로, 이전 bot 응답에
    포함된 ```sql 코드블록을 파싱해 `last_sql` 로 되돌린다.
    """
    # 0) 인사 → SQL 생략
    if GREETING_PATTERNS.search(message or ""):
        return random.choice(GREETING_RESPONSES)

    # 0b) 범위 외 휴리스틱
    if OUT_OF_SCOPE_HINTS.search(message or ""):
        return (
            "죄송합니다. 이 에이전트는 병원 DB (환자·의사·진료·진단·진료과)에 대한 질문만 다룰 수 있습니다.\n"
            "예: '지난달 완료된 진료 건수는?', '진료과별 의사 수를 보여줘' 등을 시도해 보세요."
        )

    # 1) history → history_text + last_sql 복원
    history_text = ""
    last_sql = ""
    for turn in (history or [])[-5:]:
        if isinstance(turn, dict):
            role = turn.get("role", "user")
            content = turn.get("content", "") or ""
            history_text += f"{role}: {content[:200]}\n"
            if role == "assistant":
                m = re.search(r"```sql\s*\n(.*?)\n```", content, re.DOTALL)
                if m:
                    last_sql = m.group(1).strip()
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_msg, bot_msg = turn
            history_text += f"사용자: {user_msg}\n시스템: {(bot_msg or '')[:200]}\n"
            m = re.search(r"```sql\s*\n(.*?)\n```", bot_msg or "", re.DOTALL)
            if m:
                last_sql = m.group(1).strip()

    # 2) SQL 생성 → 실행 → 요약
    try:
        sql = generate_sql(message, history_text, last_sql)
        result = safe_execute(sql)
        if result.startswith("[blocked]") or result.startswith("[error]"):
            return result
        answer = summarize_result(message, sql, result)
        return (
            f"{answer}\n\n---\n실행된 SQL:\n```sql\n{sql}\n```\n\n원본 결과:\n{result}"
        )
    except Exception as e:
        return f"처리 중 오류가 발생했습니다: {type(e).__name__}: {str(e)[:120]}"


# 핸들러 자체를 로컬 테스트 (Gradio UI 띄우지 않고도 확인 가능)
print(hospital_chat("안녕하세요", history=None))
print()
print(hospital_chat("오늘 날씨 어때?", history=None))

## 8. Gradio ChatInterface 실행

아래 셀을 실행하면 `Running on public URL: https://xxxxx.gradio.live` 가 출력됩니다. 그 URL 을 브라우저에서 열면 채팅 UI 가 보이고, 다른 수강생에게 공유해 교차 체험이 가능합니다.

> Gradio 4.x 에서는 `retry_btn` / `undo_btn` / `clear_btn` 가 동작합니다. 5.x 로 올리면 이 파라미터들은 제거되었으므로 삭제해야 합니다.

In [ ]:
# ============================================================
# 10. Gradio ChatInterface — 병원 DB AI 상담사
# ============================================================
demo = gr.ChatInterface(
    fn=hospital_chat,
    title="병원 DB AI 상담사",
    description="자연어로 병원 데이터베이스에 질문하세요. 환자, 의사, 진료 기록을 분석합니다.",
    examples=[
        "현재 등록된 환자 수는?",
        "진료과별 의사 수를 보여줘",
        "지난 3개월간 가장 많이 방문한 환자 Top 5 는?",
        "응급 진료 건수와 평균 비용은?",
    ],
    theme=gr.themes.Soft(),
    # Gradio 4.x 전용 파라미터 (5.x 에서는 제거됨)
    retry_btn="다시 시도",
    undo_btn="실행 취소",
    clear_btn="대화 초기화",
)

demo.launch(share=True)

## 9. (선택) 고급 UI — `gr.Blocks` + 커스텀 CSS

사이드바에 예시 질문·주의사항을 배치하고, 하단에 전송/초기화 버튼을 분리한 블록 UI 예시입니다. 같은 `hospital_chat` 핸들러를 재사용합니다.

In [ ]:
# ============================================================
# 11. (선택) 고급 UI — gr.Blocks
# ============================================================
custom_css = """
.gradio-container {
    max-width: 900px !important;
    margin: auto !important;
}
.message-bubble-border {
    border-radius: 12px !important;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="병원 DB 상담사") as advanced_demo:
    gr.Markdown("# 병원 DB AI 상담사")
    gr.Markdown("자연어로 병원 데이터를 분석하세요. SQL 을 자동으로 생성·실행합니다.")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=500,
                bubble_full_width=False,  # Gradio 4.x 전용
                show_label=False,
            )
            msg = gr.Textbox(
                placeholder="질문을 입력하세요... (예: 남성 환자 수는?)",
                show_label=False,
                scale=4,
            )
            with gr.Row():
                submit_btn = gr.Button("전송", variant="primary")
                clear_btn  = gr.Button("초기화")

        with gr.Column(scale=1):
            gr.Markdown("### 예시 질문")
            gr.Markdown(
                "- 환자 수는 몇 명?\n"
                "- 진료과별 의사 수\n"
                "- 월별 방문 추이\n"
                "- 가장 비싼 진료 5 건\n"
                "- 중증 진단 환자 목록"
            )
            gr.Markdown("### 주의사항")
            gr.Markdown(
                "- SELECT 쿼리만 가능\n"
                "- 데이터 수정/삭제 불가\n"
                "- 결과는 최대 1000 행"
            )

    def respond(message, chat_history):
        response = hospital_chat(message, chat_history)
        chat_history.append((message, response))
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    submit_btn.click(respond, [msg, chatbot], [msg, chatbot])
    clear_btn.click(lambda: None, None, chatbot, queue=False)

# 주석 해제하면 Blocks 버전 UI 가 뜹니다.
# advanced_demo.launch(share=True)
print("advanced_demo 정의 완료. launch(share=True) 로 실행할 수 있습니다.")

## 10. 수강생 간 교차 체험 안내

각자 생성한 `https://xxxxx.gradio.live` URL 을 짝꿍과 교환하고, 서로의 앱에서 질문 3 개씩 테스트해 보세요. "내 에이전트가 왜 이 질문에서 실패하지?" 를 **다른 사람의 시선**으로 발견하는 것이 가장 빠른 디버깅 방법입니다.

In [ ]:
# 교차 체험 진행 스크립트 (출력만 하는 셀)
print("""
교차 체험 진행 (약 10 분)

1. 본인 Gradio URL 을 강의 채팅방에 공유
2. 다른 학생 2 명의 URL 에 접속
3. 각 앱에서 질문 3 개씩 테스트
   - Easy / Medium / Hard 1 개씩
   - 일부러 모호한 질문도 1 개
4. 발견한 문제점을 정리해 해당 학생에게 피드백
5. 기본 파라미터 auth=("id","pw") 로 잠그고 싶다면 demo.launch(share=True, auth=("team","pw"))
""")

## 11. 과제 #2 — 스키마 + 시드 데이터 배포 (Day 3 시작 때 제출)

Day 3 시작(13H) 까지 본인 프로젝트 DB 를 **Neon 에 배포**해야 합니다. 아래 체크리스트를 지금 확인하세요.

- [ ] 제안서 스키마가 Neon 에 그대로 반영되었는가?
- [ ] 테이블당 **최소 50 행** 시드 데이터를 넣었는가?
- [ ] 모든 컬럼에 `COMMENT ON COLUMN` 을 달았는가?
- [ ] FK 제약 조건이 실제로 작동하는가?
- [ ] (선택) **read-only** DSN 또는 스크린샷으로 강사에게 확인 제출

> **주의** — 원본(read-write) DSN 을 그대로 공유하지 마세요. 공유가 필요하면 Neon 대시보드에서 `GRANT SELECT ON ALL TABLES IN SCHEMA public TO reviewer_ro;` 로 read-only 롤을 따로 만들어 쓰세요.

In [ ]:
# ============================================================
# 12. Faker 로 시드 데이터 생성 (샘플)
# ============================================================
# !pip install -q faker   # 필요 시 주석 해제
# from faker import Faker
# fake = Faker("ko_KR")
#
# customers = []
# for _ in range(50):
#     customers.append({
#         "name":       fake.name(),
#         "email":      fake.email(),
#         "phone":      fake.phone_number(),
#         "address":    fake.address(),
#         "created_at": fake.date_between("-1y", "today"),
#     })
#
# df = pd.DataFrame(customers)
# print(df.head(10))
#
# # DB 삽입은 FK 부모 테이블부터:
# # df.to_sql("customers", engine, if_exists="append", index=False)
print("Faker 예시 코드는 위에 주석 처리되어 있습니다. 본인 스키마 구조에 맞춰 복사해 쓰세요.")

## 실습 과제

다음 3 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. 5턴 이상 연속 대화
**5턴 이상 연속 대화를 시도하세요.**

아래 시나리오를 참고하여 5턴 이상의 연속 대화를 진행해보세요. 맥락이 잘 유지되는지 확인합니다.

시나리오 예시 (참고용):

1. "여성 환자는 몇 명인가요?"
2. "그 중에 30세 미만은?"
3. "그 사람들 중 응급 진료를 받은 적 있는 사람은?"
4. "그 환자들의 진단명을 보여줘"
5. "그 중 중증(severe) 진단은?"

**확인할 점:**

- "그 중에", "그 사람들" 표현이 올바르게 해석되는가?
- 3턴 이후에도 처음 조건(여성)이 유지되는가?
- 5턴에서 맥락이 흐려지거나 잘못된 SQL이 생성되지는 않는가?

_힌트: `HospitalChatbot(engine, schema_info)` 인스턴스를 만들고 질문을 리스트로 정리한 뒤 for 루프로 `bot.chat(q)` 를 호출해 출력하세요. 정답 코드는 숨겨져 있습니다 -- 본인이 직접 시나리오를 짜서 돌려 보아야 맥락 유지가 어디서 깨지는지 보입니다._

### 2. UNION 차단 패턴 추가
**UNION 차단 패턴을 SQLGuardrail에 추가하세요.**

`SQLGuardrail` 을 상속한 `EnhancedSQLGuardrail` 클래스를 만들고, `BLOCKED_PATTERNS` 정규식에 `UNION SELECT` / `UNION ALL SELECT` 를 잡아내는 항목을 추가하세요.

아래 3가지 케이스로 동작을 검증하세요:

1. 정상 쿼리: `SELECT name FROM patients WHERE gender = 'M'` -> 통과
2. UNION 인젝션: `SELECT name FROM patients UNION SELECT password FROM pg_roles` -> 차단
3. UNION ALL 인젝션: `SELECT name FROM patients UNION ALL SELECT table_name FROM information_schema.tables` -> 차단

_힌트: 기존 `SQLGuardrail.BLOCKED_PATTERNS` 정규식을 그대로 복사한 뒤, alternation(`|`) 으로 `\bUNION\b\s+(ALL\s+)?SELECT` 같은 패턴을 한 줄 끼워 넣으면 됩니다. 정답 코드는 숨겨져 있습니다 -- 정규식을 직접 작성해 보아야 escape/대소문자 처리가 손에 익습니다._

### 3. 인사 감지 기능 추가
**인사 감지 기능을 추가하세요.**

"감사합니다", "안녕", "고마워요" 같은 인사가 오면 SQL을 생성하지 않고 인사로 응답하는 기능을 `hospital_chat`에 추가합니다.

구현 가이드:

1. 인사 키워드(감사합니다 / 고마워 / 안녕 / 수고 / 반갑 / 잘 부탁 / 좋은 하루 / 화이팅 등)를 잡는 정규식 `GREETING_PATTERNS` 를 만드세요
2. 응답 후보 문자열 리스트 `GREETING_RESPONSES` 를 정의하세요 (3~5개 권장)
3. `hospital_chat` 을 복사해 `hospital_chat_v2` 를 만들고, 함수 가장 첫 줄에서 인사가 매칭되면 `random.choice(GREETING_RESPONSES)` 를 즉시 return 하도록 분기하세요
4. 매칭되지 않을 때만 기존 SQL 생성/실행 로직으로 흘러가야 합니다
5. 새 핸들러를 `gr.ChatInterface` 로 감싸 `examples=["안녕하세요!", "환자 수는?", "감사합니다!"]` 로 동작을 확인하세요

_힌트: 위쪽 "hospital_chat 핸들러" 셀을 베이스로 두고, **함수 본문 시작 부분에 if 분기 한 줄만** 추가하면 됩니다. 정답 코드는 숨겨져 있으니, 먼저 본인이 짠 정규식이 "안녕하세요" / "감사해요" / "환자 수는?" 세 문장을 어떻게 구분하는지 직접 테스트해 보세요._


In [ ]:
# ============================================================
# 실습 과제 — 멀티턴 대화 / UNION 차단 / 인사 감지
# ============================================================

# 실습 1: 5턴 이상 연속 대화
# TODO: HospitalChatbot 인스턴스를 만들고 5턴 시나리오를 for 루프로 실행해 맥락 유지 여부를 확인하세요.
# 여기에 구현하세요.


# 실습 2: UNION 차단 패턴 추가 (EnhancedSQLGuardrail)
# TODO: SQLGuardrail 을 상속하고 BLOCKED_PATTERNS 에 UNION (ALL )?SELECT 패턴을 추가한 뒤 3가지 케이스로 검증하세요.
# 여기에 구현하세요.


# 실습 3: 인사 감지 기능 추가 (hospital_chat_v2)
# TODO: GREETING_PATTERNS 정규식과 GREETING_RESPONSES 리스트를 만들고, hospital_chat_v2 함수 첫 줄에서 매칭 시 random.choice 로 즉시 return 하세요.
# 여기에 구현하세요.


## 11~12H 핵심 정리

- **멀티턴** = 이전 대화를 기억하는 채팅. "대화 메모장(`ChatState`)" 또는 Gradio `history` 가 그 역할.
- **가드레일** = 첫 번째 경비원. SELECT 만 허용·위험 키워드 차단·`LIMIT` 자동 주입. 단 정규식 가드레일은 **보조**일 뿐 — 프로덕션은 DB read-only 롤이 본선입니다.
- **에러 복구** = 에러 메시지를 프롬프트에 되먹여 재시도. LangGraph 루프의 씨앗입니다.
- **Gradio** = `ChatInterface(fn=..., examples=..., theme=...)` 5 줄로 채팅 UI 완성. Colab 에서 `share=True` 로 공개 URL 자동 생성.

## 다음 노트북에서는…

Day 3 의 첫 노트북 **`10_vanna_intro.ipynb`** 에서 오늘 수동으로 짰던 "스키마 주입 + Few-shot + 재시도" 를 **자동화** 하는 Vanna.ai 를 소개합니다. Vanna 는 ChromaDB 에 (DDL / Documentation / SQL Pairs) 3 종 학습 자산을 쌓아 두고, 질문이 오면 관련 자산만 RAG 로 꺼내 Text-to-SQL 을 수행합니다. Day 3 13~14H 에서 본인 프로젝트 DB 에 Vanna 를 학습시켜 정확도를 끌어올립니다.